In [1]:
import torch
import torchvision
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader

In [2]:
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)

PyTorch: 2.14.0+cpu
Torchvision: 0.29.0+cpu


In [3]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

In [4]:
class AcneDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):

        row = self.dataframe.iloc[index]

        image = Image.open(row["image_path"]).convert("RGB")
        label = int(row["acne_level"])

        if self.transform:
            image = self.transform(image)

        return image, label

In [10]:
from pathlib import Path
import pandas as pd

processed_dir = Path("datasets") / "processed"

print(processed_dir.resolve())

C:\Users\NuKe\OneDrive\Desktop\SKINORA\ml\datasets\processed


In [11]:
clean_df = pd.read_csv(
    processed_dir / "acne_dataset.csv"
)

print("Rows:", len(clean_df))
print(clean_df["acne_level"].value_counts().sort_index())

Rows: 1406
acne_level
0    491
1    623
2    177
3    115
Name: count, dtype: int64


In [12]:
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(
    clean_df,
    test_size=0.30,
    stratify=clean_df["acne_level"],
    random_state=42
)

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["acne_level"],
    random_state=42
)

print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

Train: 984
Validation: 211
Test: 211


In [13]:
train_files = set(train_df["filename"])
val_files = set(val_df["filename"])
test_files = set(test_df["filename"])

print("Train ∩ Validation:", len(train_files & val_files))
print("Train ∩ Test:", len(train_files & test_files))
print("Validation ∩ Test:", len(val_files & test_files))

Train ∩ Validation: 0
Train ∩ Test: 0
Validation ∩ Test: 0


In [14]:
train_df.to_csv(
    processed_dir / "train.csv",
    index=False
)

val_df.to_csv(
    processed_dir / "val.csv",
    index=False
)

test_df.to_csv(
    processed_dir / "test.csv",
    index=False
)

print("Saved successfully.")

Saved successfully.


In [15]:
for file in processed_dir.iterdir():
    print(file.name)

acne_dataset.csv
test.csv
train.csv
val.csv


In [16]:
from pathlib import Path
import pandas as pd

print("Current directory:", Path.cwd())

Current directory: C:\Users\NuKe\OneDrive\Desktop\SKINORA\ml
